# HAC Steel OCBF — Phase 0: Seismic Base Shear & Capacity Reconciliation

**ELF base shear + capacity-design demands — ASCE 7-22 (Ch. 12 & 15), AISC 360-22 / 341-22, 2025 CBC**

| field | value |
|---|---|
| Project | OAI Richmond Robotic Lab — Phase 2 |
| Job No. | DG26.0160.00 |
| Location | 1411 Harbour Way S., Richmond, CA |
| Date | 2026-07-29 |
| Subject | HAC Steel OCBF — base shear · **Rev A (Concept)** |
| Prepared | Jeffrey (Structural Intern) |
| Checked | S. Aher (pending) |

**Basis.** 2025 CBC / ASCE 7-22 (Ch. 12 & 15); AISC 360-22 & 341-22. The permit
design remains code-based (ELF, Steel OCBF, `R = 3.25`); this sheet supports the
beyond-code PBD options study for the HAC base connection. Units are carried in
kip–in–ksi via `forallpeople`; the seismic coefficients `SDS`, `Cs`, `R`, `Ie`
are expressed in units of `g`.

**Purpose.** Establish the seismic base shear for one typical HAC module and the
amplified (capacity-design) demands on the force-controlled base connection, then
reconcile them against the frame's lateral capacity and the intended base-fuse
plateau (Solution 2). This is the hand-calc anchor for Phase 0 of the nonlinear
workflow: the 2D Perform-3D pushover must reproduce `V` and the fuse plateau, and
every capacity-protected element must remain elastic against the fuse overstrength.

> **PRELIMINARY — PBD OPTIONS STUDY — NOT FOR PERMIT / CONSTRUCTION**


In [1]:
%%capture

# --- Environment setup -------------------------------------------------
!pip install handcalcs
!pip install forallpeople
!pip install numpy
!pip install pandas

import sys
import os
# Tell Python to look in the parent directory (Project_Root) for your modules
sys.path.append(os.path.abspath('..'))

import handcalcs.render                     # registers the %%render cell magic
import forallpeople as si
si.environment('structural')            # units become si.* attributes

# pull the structural units into the notebook namespace
kip, lb, inch, ft   = si.kip, si.lb, si.inch, si.ft
ksi, psi, ksf, psf  = si.ksi, si.psi, si.ksf, si.psf

# trig / roots used in the render cells below
from math import sin, cos, tan, atan, sqrt, pi

# helper modules (uploaded alongside this notebook)
from helpers.sigfigs import setup_formatting, sig   # 3-sig-fig handcalcs patch + sig() for text
from helpers.exporter import export_notebook        # absolute-path HTML/PDF export
setup_formatting(3)

# NOTE: work strictly in kip and inch (never ft) so derived forces/moments/stresses
# stay in kip / kip-in / ksi instead of collapsing to SI. Seismic coefficients
# (SDS, Cs, R, Ie, Ta) are kept as dimensionless floats.


## 1  Seismic Design Parameters (ASCE 7-22 / 2025 CBC)

Mapped and design spectral values from the Concept BOD (Site Class D, SDC D).
`SDS` governs the short-period plateau; `Ω0` and `Ie` follow Table 15.4-1 for a
Steel OCBF designed as a nonbuilding structure similar to a building.

In [2]:
%%render params
S_S     = 1.960          # g, mapped $MCE_R$ (Ch. 22)
S_1     = 0.680          # g, mapped $MCE_R$
S_DS    = 1.310          # g, design short-period (governs)
S_D1    = 1.130          # g, design 1-second
R       = 3.250          # Steel OCBF, Table 15.4-1
Omega_0 = 2.000          # overstrength factor
I_e     = 1.000          # Risk Category II
T_a     = 0.182          # s, approx. period (Eq. 12.8-7)
W       = 54*kip         # module seismic weight (frame + MEP)


<IPython.core.display.Latex object>

## 2  Period Region Check

The spectral corner period `Ts = SD1 / SDS` locates the module on the response
spectrum.

In [3]:
%%render
T_s = S_D1 / S_DS          # spectral corner period (s)


<IPython.core.display.Latex object>

Since `Ta = 0.182 s  ≪  Ts = 0.863 s`, the module sits on the **short-period
(constant-acceleration) plateau** — it attracts near-peak spectral acceleration.
This is the physical root of the acceleration problem the PBD study addresses.

## 3  Seismic Base Shear (ELF, ASCE 7-22 §12.8)

`Cs` from Eq. 12.8-2, bracketed by the upper limit (Eq. 12.8-3) and the two lower
limits (Eqs. 12.8-5, 12.8-6). Eq. 12.8-6 applies because `S1 = 0.68 g ≥ 0.6 g`.

In [4]:
%%render
C_s     = S_DS / (R / I_e)              # Eq. 12.8-2  (governs)
C_smax  = S_D1 / (T_a * (R / I_e))      # Eq. 12.8-3  upper limit
C_smin1 = 0.044 * S_DS * I_e            # Eq. 12.8-5  lower limit
C_smin2 = 0.5 * S_1 / (R / I_e)         # Eq. 12.8-6  (S1 >= 0.6g)
V       = C_s * W                       # Eq. 12.8-1  base shear


<IPython.core.display.Latex object>

`Cs = 0.403` by Eq. 12.8-2 (`0.105 < 0.403 < 1.910`, so neither limit
governs). **Seismic base shear `V = 21.8 kip` per 26-ft module** — this is the
permit-basis demand (`R = 3.25`) and does not change; everything below is the
beyond-code reconciliation.

## 4  Vertical Seismic Load Effect

In [5]:
%%render
E_v = 0.2 * S_DS          # Eq. 12.4-4a  vertical seismic (x D)


<IPython.core.display.Latex object>

## 5  Overstrength (Capacity-Design) Demand

Force-controlled demand for the brace connections and anchorage.

In [6]:
%%render
V_o  = Omega_0 * V              # Eq. 12.4-7  force-controlled demand
V_mt = 1.5 * Omega_0 * V        # if multi-tiered OCBF (AISC 341 F1)


<IPython.core.display.Latex object>

If the stacked-X frame is classified **multi-tiered** per AISC 341 §F1,
columns, struts, and their connections use `1.5 Ω0 = 3.0` (`Vmt = 65.3 kip`).
Resolve single- vs multi-tiered before finalizing the capacity-protection
checks — it moves the demand by 50%.

## 6  Demand Path — Brace Force & Base Reactions

Per-brace equilibrium at the base: the QE components `VQE` (vertical) and `HQE`
(horizontal) resolve into the brace axial `PQE`; the amplified reactions are the
anchorage / fuse sizing forces.

In [7]:
%%render
V_QE  = 24*kip                       # per-brace vertical (uplift) QE
H_QE  = 9*kip                        # per-brace horizontal QE
theta = 69.4                         # deg, brace angle = atan($V_{QE} / H_{QE}$)
P_QE  = V_QE / sin(theta * pi/180)   # brace axial QE
P_u   = Omega_0 * P_QE               # amplified brace axial
H_u   = Omega_0 * H_QE               # amplified base shear (shear lug)
T_u   = Omega_0 * V_QE               # amplified base uplift (anchorage)


<IPython.core.display.Latex object>

**Cross-check.** The amplified base reactions (`Hu = 18 kip`, `Tu = 48 kip`
per brace) match the S0.04 short-direction values (18 / 48). These are the
anchorage / fuse sizing forces; **David's RISA model governs** over this hand
distribution — use the hand calc to confirm RISA is sane, not to replace it.

## 7  Brace Capacity (frame overstrength vs. demand)

Representative brace `HSS7×4×3/8` (confirm `Ag`, `r` from the member schedule /
AISC Manual). Expected tension yield per AISC 341; compression per AISC 360 §E3.

In [8]:
%%render params
R_y  = 1.400             # Ry, expected/nominal yield ratio (A500 Gr.C HSS)
F_y  = 50*ksi            # nominal yield
A_g  = 6.900*inch**2     # gross area
K    = 1.000             # effective length factor
L_b  = 110.6*inch        # brace unbraced length
r_b  = 1.550*inch        # governing radius of gyration
E    = 29000*ksi         # modulus


<IPython.core.display.Latex object>

In [9]:
%%render
T_yield  = R_y * F_y * A_g                     # expected tension yield (AISC 341)
lambda_c = K * L_b / r_b                        # slenderness KL/r
F_e      = pi**2 * E / lambda_c**2              # Eq. E3-4  elastic buckling stress
F_cr     = 0.658**(F_y / F_e) * F_y             # Eq. E3-2  inelastic buckling
P_n      = F_cr * A_g                           # nominal compression capacity


<IPython.core.display.Latex object>

Both brace capacities (`Tyield ≈ 483 kip`, `Pn ≈ 238 kip`) dwarf the
`≈ 26 kip` demand. The frame's true lateral overstrength is enormous, so a bare
(Solution 1) frame stays elastic to a very high base shear — which is exactly why
a base fuse pays off: it places a controlled yield far below the frame's brute
capacity and caps what reaches the servers.

## 8  Base-Fuse Plateau & Acceleration Ceiling

The heel-group yield sets a force plateau `Vpl`; the plateau base-shear
coefficient `acap = Vpl / W` is the ceiling on floor acceleration.

In [10]:
%%render params
n_t  = 4                 # number of heel-group fuse rods
A_sh = 0.300*inch**2     # reduced-shank area per rod (PRELIM)
F_yr = 105*ksi           # rod yield (F1554 Gr. 105)
V_pl = 20*kip            # target plateau base shear (PRELIM, see Phase 3)


<IPython.core.display.Latex object>

In [11]:
%%render
T_fuse = n_t * A_sh * F_yr          # heel-group yield (PRELIM, see Phase 3)
a_cap  = V_pl / W                    # g, plateau base-shear coefficient
ratio  = S_DS / (V_pl / W)           # accel reduction vs. elastic frame


<IPython.core.display.Latex object>

`acap = Vpl/W ≈ 0.37 g` at the plateau, versus `≈ SDS = 1.31 g` transmitted
by an essentially-elastic frame — a **≈ 3.5× cut in peak floor acceleration**, the
metric that drives the FEMA P-58 / SP3 comparison.

> **Preliminary:** `nt` and `Ashank` above are placeholders. The fuse backbone
> (reduced-shank area, yield force, and yield displacement over the debonded
> stretch length) is finalized in Phase 3. NLRHA quantifies the real reduction
> once the elastic spike before fuse engagement and higher-mode content are
> included.

## 9  Results Summary

In [12]:
import pandas as pd
def kv(x, u="kip"):
    return f"{sig(float(x))} {u}".strip()

rows = [
    ("Seismic base shear  V (Eq. 12.8-1)",        kv(V),        "R = 3.25 · permit basis"),
    ("Overstrength shear  Vo = Ω0·V",             kv(V_o),      "single-tier"),
    ("Multi-tier shear  Vmt = 1.5·Ω0·V",          kv(V_mt),     "if MT per AISC 341 §F1"),
    ("Amplified brace axial  Pu",                 kv(P_u),      ""),
    ("Amplified base shear  Hu (shear lug)",      kv(H_u),      "matches S0.04 (18)"),
    ("Amplified base uplift  Tu (anchorage)",     kv(T_u),      "matches S0.04 (48)"),
    ("Brace tension yield  Tyield",               kv(T_yield),  "expected (AISC 341)"),
    ("Brace compression  Pn",                     kv(P_n),      "AISC 360 §E3"),
    ("Fuse plateau coeff  acap = Vpl/W",          kv(a_cap, "g"),"vs SDS = 1.31 g"),
    ("Accel reduction  SDS / (Vpl/W)",            f"{sig(float(ratio))}\u00d7", "peak floor-accel cut"),
]
df = pd.DataFrame(rows, columns=["Quantity", "Value", "Note"])
df


,Quantity,Value,Note
0,Seismic base shear V (Eq. 12.8-1),21.8 kip,R = 3.25 · permit basis
1,Overstrength shear Vo = Ω0·V,43.5 kip,single-tier
2,Multi-tier shear Vmt = 1.5·Ω0·V,65.3 kip,if MT per AISC 341 §F1
3,Amplified brace axial Pu,51.3 kip,
4,Amplified base shear Hu (shear lug),18 kip,matches S0.04 (18)
5,Amplified base uplift Tu (anchorage),48 kip,matches S0.04 (48)
6,Brace tension yield Tyield,483 kip,expected (AISC 341)
7,Brace compression Pn,238 kip,AISC 360 §E3
8,Fuse plateau coeff acap = Vpl/W,0.37 g,vs SDS = 1.31 g
9,Accel reduction SDS / (Vpl/W),3.54×,peak floor-accel cut


## 10  Reconciliation Targets for the 2D Perform-3D Model

The nonlinear model must reproduce this hand calc before any dynamic result is
trusted:

1. **Elastic period `T1 ≈ 0.18 s`** (modal check — if off, mass or stiffness is wrong).
2. **First yield occurs in the fuse**, at base shear `≈ Vpl` (18–22 kip).
3. **Pushover plateaus flat** (elastic–plastic); a flag shape means re-centering leaked in.
4. **Base-shear ceiling `≤` fuse overstrength** (`nt·Ashank·Rt·Fu`) — nothing above the fuse yields.
5. **Usage ratios `< 1.0`** on braces, gusset, base plate, shear lug, and ACI 318-19
   Ch. 17 anchorage, all checked against the fuse overstrength (capacity-design hierarchy).


## 11  References & Export

**References:** ASCE 7-22 §12.8, §12.4, Table 15.4-1; AISC 360-22 §E3;
AISC 341-22 §F1; ACI 318-19 Ch. 17. Reaction cross-check per drawing S0.04.
Representative brace HSS7×4×3/8 (confirm `Ag`, `r` from the member schedule /
AISC Manual). Prepared with `handcalcs` + `forallpeople`.

Export via the uploaded helper (writes to the project `outputs/` folder):

In [13]:
# export_notebook("html")   # or "pdf" (WeasyPrint backend on Colab)
export_notebook("html")


Detected active notebook: 'Richmond_HAC_OCBF_BaseShear_CapacityReconciliation_v01.ipynb'
Exporting to HTML in: C:\Users\jeffreyda\hand_calcs\outputs
Success! Your file is ready in the 'outputs' folder.
